In [ ]:
# Estudiante: Masiel Aguilar Ameller
# Codigo: 87770

from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.style import WD_STYLE_TYPE
import pandas as pd
from datetime import datetime
import os
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen
import warnings
warnings.filterwarnings('ignore')

class BBBPWordReportGenerator:
    def __init__(self, csv_path):
        """
        Initialize the BBBP Word Report Generator
        
        Args:
            csv_path (str): Path to the BBBP.csv file
        """
        self.csv_path = csv_path
        self.data = None
        self.load_data()
        
    def load_data(self):
        """Load the BBBP dataset from CSV file"""
        try:
            self.data = pd.read_csv(self.csv_path)
            print(f"Dataset loaded successfully: {len(self.data)} compounds")
        except Exception as e:
            print(f"Error loading dataset: {e}")
    
    def calculate_molecular_properties(self, smiles):
        """Calculate molecular properties from SMILES string"""
        try:
            from rdkit import RDLogger
            RDLogger.DisableLog('rdApp.*')
            
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return self.get_default_properties()
            
            properties = {
                'molecular_weight': round(Descriptors.MolWt(mol), 2),
                'logp': round(Crippen.MolLogP(mol), 2),
                'hbd': Descriptors.NumHDonors(mol),
                'hba': Descriptors.NumHAcceptors(mol),
                'tpsa': round(Descriptors.TPSA(mol), 2),
                'rotatable_bonds': Descriptors.NumRotatableBonds(mol),
                'aromatic_rings': rdMolDescriptors.CalcNumAromaticRings(mol),
                'heavy_atoms': Descriptors.HeavyAtomCount(mol),
                'complexity': round(Descriptors.BertzCT(mol), 2)
            }
            
            properties['lipinski_violations'] = sum([
                properties['molecular_weight'] > 500,
                properties['logp'] > 5,
                properties['hbd'] > 5,
                properties['hba'] > 10
            ])
            
            return properties
            
        except Exception as e:
            return self.get_default_properties()
    
    def get_default_properties(self):
        """Return default properties when calculation fails"""
        return {
            'molecular_weight': 'N/A', 'logp': 'N/A', 'hbd': 'N/A', 'hba': 'N/A',
            'tpsa': 'N/A', 'rotatable_bonds': 'N/A', 'aromatic_rings': 'N/A',
            'heavy_atoms': 'N/A', 'complexity': 'N/A', 'lipinski_violations': 'N/A'
        }
    
    def assess_bbb_penetration_factors(self, properties, p_np):
        """Assess factors affecting blood-brain barrier penetration"""
        assessment = {
            'penetration_status': 'Penetrante' if p_np == 1 else 'No penetrante',
            'favorable_factors': [],
            'unfavorable_factors': [],
            'overall_assessment': '',
            'confidence_level': 'Medio'
        }
        
        if properties['molecular_weight'] != 'N/A':
            if properties['molecular_weight'] < 400:
                assessment['favorable_factors'].append('Peso molecular bajo (< 400 Da) favorece la penetración BHE')
            elif properties['molecular_weight'] > 500:
                assessment['unfavorable_factors'].append('Peso molecular alto (> 500 Da) dificulta la penetración BHE')
            
            if properties['logp'] != 'N/A':
                if 1 <= properties['logp'] <= 3:
                    assessment['favorable_factors'].append('Lipofilia óptima (LogP 1-3) para penetración BHE')
                elif properties['logp'] < 1:
                    assessment['unfavorable_factors'].append('Baja lipofilia puede limitar la penetración BHE')
                elif properties['logp'] > 5:
                    assessment['unfavorable_factors'].append('Alta lipofilia puede causar unión inespecífica')
            
            if properties['tpsa'] != 'N/A':
                if properties['tpsa'] < 60:
                    assessment['favorable_factors'].append('Área polar superficial baja (< 60 Ų) favorece la penetración BHE')
                elif properties['tpsa'] > 90:
                    assessment['unfavorable_factors'].append('Área polar superficial alta (> 90 Ų) dificulta la penetración BHE')
        
        favorable_count = len(assessment['favorable_factors'])
        unfavorable_count = len(assessment['unfavorable_factors'])
        
        if favorable_count > unfavorable_count:
            assessment['overall_assessment'] = 'Las propiedades moleculares favorecen generalmente la penetración BHE'
            assessment['confidence_level'] = 'Alto' if favorable_count >= 3 else 'Medio'
        elif unfavorable_count > favorable_count:
            assessment['overall_assessment'] = 'Las propiedades moleculares dificultan generalmente la penetración BHE'
            assessment['confidence_level'] = 'Alto' if unfavorable_count >= 3 else 'Medio'
        else:
            assessment['overall_assessment'] = 'Propiedades moleculares mixtas - penetración BHE incierta'
            assessment['confidence_level'] = 'Bajo'
        
        return assessment
    
    def create_word_document(self, compound_data, index):
        """Create a Word document for a single compound"""
        doc = Document()
        
        # Set document margins
        sections = doc.sections
        for section in sections:
            section.top_margin = Inches(1)
            section.bottom_margin = Inches(1)
            section.left_margin = Inches(1)
            section.right_margin = Inches(1)
        
        # Title
        title = doc.add_heading('INFORME DE ANÁLISIS DE PENETRACIÓN BARRERA HEMATOENCEFÁLICA', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Subtitle
        subtitle = doc.add_paragraph()
        subtitle_run = subtitle.add_run(f'Reporte ID: BBBP_{index:04d}')
        subtitle_run.bold = True
        subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Date
        date_para = doc.add_paragraph()
        date_run = date_para.add_run(f'Generado: {datetime.now().strftime("%d/%m/%Y %H:%M:%S")}')
        date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        doc.add_paragraph()  # Space
        
        # Get compound data
        name = compound_data.get('name', f'Compuesto_{index}')
        smiles = compound_data.get('smiles', '')
        p_np = compound_data.get('p_np', 0)
        
        # Calculate properties
        properties = self.calculate_molecular_properties(smiles)
        assessment = self.assess_bbb_penetration_factors(properties, p_np)
        
        # Compound identification section
        doc.add_heading('IDENTIFICACIÓN DEL COMPUESTO', level=1)
        
        table1 = doc.add_table(rows=3, cols=2)
        table1.style = 'Table Grid'
        
        table1.cell(0, 0).text = 'Nombre'
        table1.cell(0, 1).text = str(name)
        table1.cell(1, 0).text = 'SMILES'
        table1.cell(1, 1).text = str(smiles)
        table1.cell(2, 0).text = 'Índice en Dataset'
        table1.cell(2, 1).text = str(index)
        
        doc.add_paragraph()
        
        # BBB Penetration Status
        doc.add_heading('ESTADO DE PENETRACIÓN BARRERA HEMATOENCEFÁLICA', level=1)
        
        table2 = doc.add_table(rows=2, cols=2)
        table2.style = 'Table Grid'
        
        table2.cell(0, 0).text = 'Clasificación'
        table2.cell(0, 1).text = assessment['penetration_status']
        table2.cell(1, 0).text = 'Nivel de Confianza'
        table2.cell(1, 1).text = assessment['confidence_level']
        
        doc.add_paragraph()
        
        # Molecular Properties
        doc.add_heading('ANÁLISIS DE PROPIEDADES MOLECULARES', level=1)
        
        table3 = doc.add_table(rows=10, cols=2)
        table3.style = 'Table Grid'
        
        properties_list = [
            ('Peso Molecular', f'{properties["molecular_weight"]} Da'),
            ('Lipofilia (LogP)', str(properties['logp'])),
            ('Donadores de Enlace H', str(properties['hbd'])),
            ('Aceptores de Enlace H', str(properties['hba'])),
            ('Área Polar Superficial', f'{properties["tpsa"]} Ų'),
            ('Enlaces Rotables', str(properties['rotatable_bonds'])),
            ('Anillos Aromáticos', str(properties['aromatic_rings'])),
            ('Átomos Pesados', str(properties['heavy_atoms'])),
            ('Complejidad Molecular', str(properties['complexity'])),
            ('Violaciones Regla Lipinski', str(properties['lipinski_violations']))
        ]
        
        for i, (prop, value) in enumerate(properties_list):
            table3.cell(i, 0).text = prop
            table3.cell(i, 1).text = value
        
        doc.add_paragraph()
        
        # Assessment
        doc.add_heading('EVALUACIÓN DE PENETRACIÓN BHE', level=1)
        
        doc.add_paragraph().add_run('Evaluación General: ').bold = True
        doc.add_paragraph(assessment['overall_assessment'])
        
        doc.add_paragraph().add_run('Factores Favorables:').bold = True
        if assessment['favorable_factors']:
            for factor in assessment['favorable_factors']:
                doc.add_paragraph(f'• {factor}', style='List Bullet')
        else:
            doc.add_paragraph('• No se identificaron factores favorables significativos')
        
        doc.add_paragraph().add_run('Factores Desfavorables:').bold = True
        if assessment['unfavorable_factors']:
            for factor in assessment['unfavorable_factors']:
                doc.add_paragraph(f'• {factor}', style='List Bullet')
        else:
            doc.add_paragraph('• No se identificaron factores desfavorables significativos')
        
        doc.add_paragraph()
        
        # Clinical relevance
        doc.add_heading('RELEVANCIA CLÍNICA', level=1)
        
        if p_np == 1:
            clinical_text = """Este compuesto demuestra capacidad de penetración de la barrera hematoencefálica:
• Potencialmente adecuado para el desarrollo de fármacos del SNC
• Probable que alcance el tejido cerebral después de administración sistémica
• Importante monitorear efectos secundarios del SNC si se usa terapéuticamente
• Valioso para investigación de fármacos neurológicos y psiquiátricos"""
        else:
            clinical_text = """Este compuesto no penetra eficazmente la barrera hematoencefálica:
• Improbable que cause efectos secundarios en el SNC
• No adecuado para aplicaciones terapéuticas dirigidas al cerebro
• Puede ser preferido para dianas farmacológicas periféricas
• Podría servir como punto de partida para optimización de penetración BHE"""
        
        doc.add_paragraph(clinical_text)
        
        doc.add_paragraph()
        
        # Recommendations
        doc.add_heading('RECOMENDACIONES DE INVESTIGACIÓN', level=1)
        
        if p_np == 1:
            recommendations = """• Investigar farmacocinética del SNC y distribución cerebral
• Evaluar posibles efectos neurológicos
• Considerar para desarrollo terapéutico de enfermedades del SNC
• Monitorear interacciones con transportadores de barrera hematoencefálica"""
        else:
            recommendations = """• Considerar modificaciones estructurales para mejorar penetración BHE si se desea actividad en SNC
• Investigar farmacocinética periférica
• Evaluar para aplicaciones terapéuticas no relacionadas con SNC
• Estudiar como control negativo en estudios de penetración BHE"""
        
        doc.add_paragraph(recommendations)
        
        return doc
    
    def generate_word_reports(self, num_reports=200, output_dir='bbbp_word_reports'):
        """Generate Word reports for multiple compounds"""
        if self.data is None:
            print("No hay datos cargados. Verifique la ruta del archivo CSV.")
            return
        
        os.makedirs(output_dir, exist_ok=True)
        num_reports = min(num_reports, len(self.data))
        
        print(f"Generando {num_reports} informes en formato Word...")
        
        for i in range(num_reports):
            try:
                doc = self.create_word_document(self.data.iloc[i], i)
                
                # Clean filename
                compound_name = self.data.iloc[i].get('name', f'Compuesto_{i}')
                clean_name = compound_name.replace('/', '_').replace('\\', '_').replace(':', '_').replace('*', '_').replace('?', '_').replace('"', '').replace('<', '_').replace('>', '_').replace('|', '_').replace(' ', '_')
                
                filename = f"BBBP_Informe_{i:04d}_{clean_name}.docx"
                filepath = os.path.join(output_dir, filename)
                
                doc.save(filepath)
                
                if (i + 1) % 25 == 0:
                    print(f"Generados {i + 1} informes...")
                    
            except Exception as e:
                print(f"Error generando informe para compuesto {i}: {e}")
        
        print(f"\n¡Completado! Generados {num_reports} informes en formato Word")
        print(f"Informes guardados en '{output_dir}' directorio")

# Uso
if __name__ == "__main__":
    try:
        generator = BBBPWordReportGenerator("BBBP.csv")
        generator.generate_word_reports(num_reports=50, output_dir='bbbp_word_reports')  # Empezar con 50
        
        print("\n" + "="*60)
        print("¡GENERACIÓN DE INFORMES WORD COMPLETADA!")
        print("="*60)
        
    except Exception as e:
        print(f"Error: {e}")
        print("\nPara usar este generador, instale python-docx:")
        print("pip install python-docx")

Dataset loaded successfully: 2050 compounds
Generando 50 informes en formato Word...
Generados 25 informes...
Generados 50 informes...

¡Completado! Generados 50 informes en formato Word
Informes guardados en 'bbbp_word_reports' directorio

¡GENERACIÓN DE INFORMES WORD COMPLETADA!
